# ACC Wide Net — cap-relative component fingerprint for all 10,115 problems

**What this does, and why it is not another deep search.**

Deep search on one row answers one row. This answers the board. Two problems whose
cap-bounded components coincide are the *same problem*: one solve pays for both.
`ac-02842` and `ac-06556` share a cap-19 component of 23,609,984 states. Nobody has
swept the pool for that, and the class-size histogram tells you where the multiplier
is **before** you spend hours climbing anything.

**The trick that makes it cheap.** You cannot enumerate every component: the trivial
component `T_16` has 129,247,304 states and most easy rows live inside it. So enumerate
with a *node budget*:

| outcome | meaning |
|---|---|
| closes under budget | small isolated component — a **hard** row, and its size is a fingerprint |
| blows past budget | the giant class — record and move on |

Work per row is bounded, the sweep is embarrassingly parallel, and the interesting
rows are exactly the cheap ones.

**Equal size is necessary, not sufficient.** Measured counterexample: `ac-00945` and
`ac-05233` have identical size (386,320), identical length profile, identical degree
sequence — and set intersection **zero**. Isomorphic but disjoint. So this emits
*candidate* classes; `--verify` intersects the real state sets.

**Validated** against three independently certified facts:
`ac-02842` cap 17 = 31,240 · cap 18 = 106,800 · `ac-00738` cap 21 = 189,944.

## 1. Get the code and the pool

In [ ]:
# Upload acc_fingerprint.py and ac.jsonl (the 10,115-problem pool), or fetch them
# from wherever you keep them. Both are plain files; no install step, stdlib only.
from google.colab import files   # skip this cell on CoCalc / local
up = files.upload()              # select acc_fingerprint.py and ac.jsonl
print(sorted(up))

## 2. Self-check — reproduce the certified facts before trusting any output

In [ ]:
import sys; sys.path.insert(0, '.')
from acc_fingerprint import parse_pool, component
rows = dict(parse_pool('ac.jsonl'))
print(f'pool: {len(rows)} problems')
for cid, cap, expect in [('ac-02842',17,31240), ('ac-02842',18,106800), ('ac-00738',21,189944)]:
    size, closed, _ = component(rows[cid], cap, 5_000_000)
    print(f'{cid} cap={cap}: {size:>8}  closed={closed}  expected {expect:>8}  '
          f"{'MATCH' if size==expect else 'MISMATCH — STOP'}")

## 3. The sweep

`--delta 4` sets each row's cap to its **own** starting length + 4. A global cap is
wrong here: starting lengths run 13–40, so a fixed cap of 16 rejects most of the pool
before doing any work. Sizes are comparable within an `|r|` band, which is where
classes live anyway — family members share `|r|`.

Resumable: re-running skips rows already written, so a Colab timeout costs nothing.
Raise `--budget` for a finer split between *small* and *giant*; lower it to go faster.

In [ ]:
!python acc_fingerprint.py --pool ac.jsonl --delta 4 --budget 150000 \
    --out fp_d4.jsonl --workers 4

## 4. The histogram — where the multiplier is

In [ ]:
!python acc_fingerprint.py --pool ac.jsonl --report fp_d4.jsonl

## 5. Verify a candidate class before acting on it

This is the step that separates a real class from a coincidence. It enumerates the
actual state sets and intersects them. `IDENTICAL=True` means one solve genuinely
pays for every member; `intersection=0` means isomorphic-but-disjoint and there is
**no** transfer without an explicit symmetry map compiled to primitive moves.

In [ ]:
# put the size of the largest candidate class from step 4 here
!python acc_fingerprint.py --pool ac.jsonl --verify fp_d4.jsonl --size 26664

## 6. Plot it

In [ ]:
import json, collections
import matplotlib.pyplot as plt
recs = [json.loads(l) for l in open('fp_d4.jsonl')]
closed = [r for r in recs if r.get('status') == 'closed']
by = collections.defaultdict(list)
for r in closed: by[(r['ini'], r['size'])].append(r['challenge_id'])
sizes = sorted(collections.Counter(len(v) for v in by.values()).items())
print('class size (rows per candidate class) -> how many such classes:', sizes)
fig, ax = plt.subplots(figsize=(9,4))
ax.bar([s for s,_ in sizes], [c for _,c in sizes])
ax.set_xlabel('rows sharing one (|r|, component size)')
ax.set_ylabel('number of candidate classes')
ax.set_title('Candidate class sizes — the multiplier map')
ax.set_yscale('log'); plt.tight_layout(); plt.show()